# Gilbert Owen

# 2802432220

# DTSC6008001 – Text Mining

# 1. Data Collection

### A. Objective

**Target**: Our target is 5000 news from 5 specific news domain.

**Use** : "finance.detik.com" for finance, "inet.detik.com" for technology, "sport.detik.com" for sports, "health.detik.com" for health, "hot.detik.com" for entertainment.

**Collect via** : Web Scrapping

**Constraint** : Records containing the main article text

**Disclaimer** : data source, scraping method/tools, data collection period, number of collected records, data fields are on section 11.





### B. Data Source

**Site :** the website that is gonna be target of scrapping is **[ detik.com ]** - It is a major Indonesian news portal which has channel subdomains that is organized, also ready-made category labels

**Why Detik.com :** We choose Detik.com because it doesnt aggresively target or forbid than the other that heavily enforce automated bot bans. Also, its robot.txt permits crawling by (`Allow: /`) on every channel that we used.

**Website and its label :** 

"finance.detik.com" for finance

"inet.detik.com" for technology

"sport.detik.com" for sports

"health.detik.com" for health"

"hot.detik.com" for entertainment.

### C. How Is The Structure of Detik.com

**AI USED**

**Index page** (list of articles for one day):
`https://<channel>/indeks?date=YYYY-MM-DD&page=N` — 20 articles per page; page 1's
pagination links tell you the last page for that day.

**Article URL:** `https://<channel>/<section>/d-<id>/<slug>`
`<id>` is a unique numeric ID, used to deduplicate. `<section>` names like `foto-news`,
`video`, `detiktv`, `grafis` are photo/video/infographic pages with little or no article
text, so they are skipped.

**Article page:** the body is every `<p>` inside `div.detail__body-text`. Title, date,
author and tags come from `<meta>` tags (`og:title`, `dtk:publishdate`, `dtk:author`,
`dtk:keywords`) — these are the same across Detik's two page templates, unlike the visible
HTML.

**Known quirk:** the index page sometimes serves a different day's page than requested
(caching issue). The fix: if a requested page returns no article IDs you haven't already
seen, request it again.

### D. Import Libraries

In [26]:
import random, re, time
from datetime import timedelta, date
from pathlib import Path
import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

### E. Helper Functions

There're functions that are going to be used to help us such as :
- fetch, it takes (a URL) and returns the URL page that is also parsed by Beautiful Soup
- collect_urls, it takes a channel base URL and one date. The function returns list of that one date article URLs
- parse_article, it takes one article URL and returns a dictionary consisting of url, title, date, author, tags, and content.


### F. The code to fetch url

In [27]:
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                         "(KHTML, like Gecko) Chrome/128.0 Safari/537.36"}
SKIP_SECTIONS = re.compile(r"foto|video|detiktv|grafis", re.I)  # photo/video/infographic pages have little text


def fetch(url, params=None, delay=1.5):
    """GET a page politely (pause after every request) and return it parsed."""
    r = requests.get(url, params=params, headers=HEADERS, timeout=20)
    time.sleep(delay)
    r.raise_for_status()
    return BeautifulSoup(r.text, "lxml")


def collect_urls(base, day, max_tries=4):
    """All article URLs a Detik channel listed on one day, e.g. base='https://finance.detik.com'."""
    pattern = re.compile(re.escape(base) + r"/([\w-]+)/d-(\d+)/[\w-]+")
    seen, urls = set(), {}
    page = last_page = 1
    while page <= last_page:
        for _ in range(max_tries):
            soup = fetch(f"{base}/indeks", params={"date": day.isoformat(), "page": page})
            links = [a["href"] for a in soup.select("a[href]")]
            ids = set(re.findall(r"/d-(\d+)/", " ".join(links)))
            if ids - seen:  # Detik sometimes returns a page we already have; if so, ask again
                break
        seen |= ids
        if page == 1:  # page 1 links to the day's last page ("1 2 3 4 5 ... 15")
            last_page = max([int(n) for h in links for n in re.findall(r"[?&]page=(\d+)", h)], default=1)
        for h in links:
            m = pattern.match(h)
            if m and not SKIP_SECTIONS.search(m.group(1)):
                urls[m.group(2)] = m.group(0)  # key = article id, so the same article isn't kept twice
        page += 1
    return list(urls.values())


def meta(soup, **attrs):
    tag = soup.find("meta", attrs=attrs)
    return tag.get("content") if tag else None


def parse_article(url):
    """Fetch one Detik article and return its fields (content = raw paragraphs, cleaned in 02_preprocessing)."""
    soup = fetch(url)
    body = soup.select_one("div.detail__body-text")
    paragraphs = [p.get_text(" ", strip=True) for p in body.find_all("p")] if body else []
    return {
        "url": url,
        "title": meta(soup, property="og:title"),
        "date": meta(soup, name="dtk:publishdate"),
        "author": meta(soup, name="dtk:author"),
        "tags": meta(soup, name="dtk:keywords"),
        "content": "\n".join(p for p in paragraphs if p),
    }

### G. Configuration of Labels and Channels

**Labels and channels** , plus:
- `START` / `END`: the date range to scrape 
- `PER_DAY`: articles to sample per label per day


In [ ]:
CHANNELS = {
    "business":      ["https://finance.detik.com"],
    "technology":    ["https://inet.detik.com"],
    "sports":        ["https://sport.detik.com", "https://sport.detik.com/sepakbola"],
    "health":        ["https://health.detik.com"],
    "entertainment": ["https://hot.detik.com"],
}
LABELS = list(CHANNELS)

START = date(2026, 6, 1)   # provisional, confirmed in Section I
END = date(2026, 8, 31)    # provisional
PER_DAY = 14               # provisional
SEED = 42
LIMIT = None                  # days per label to run: 2 = smoke test, None = full range

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)
URLS_CSV = RAW_DIR / "urls.csv"
ARTICLES_CSV = RAW_DIR / "articles.csv"
FAILED_CSV = RAW_DIR / "failed.csv"

B_LIMIT = None             # articles to download this run: 20 = smoke test, None = all of urls.csv

### H. Pilot Test

Before scraping for hours, verify the functions work end to end on a single channel and a
single day: does `collect_urls` return a sensible number of URLs, does `parse_article`
return real article text, and does the stored text match what the website actually shows?

In [29]:
test_day = date(2026, 6, 3)
test_urls = collect_urls("https://health.detik.com", test_day)
print(len(test_urls), "article URLs found")

pilot = pd.DataFrame([parse_article(u) for u in test_urls[:3]])
pilot["words"] = pilot["content"].str.split().str.len()
pilot[["date", "title", "words", "url"]]

18 article URLs found


,date,title,words,url
0,2026/06/03 20:02:28,Indonesia Jangan Cuma Jadi Pasar Terapi Gen Du...,465,https://health.detik.com/berita-detikhealth/d-...
1,2026/06/03 19:05:17,"WMO Wanti-wanti Dampak El Nino di Indonesia, S...",539,https://health.detik.com/berita-detikhealth/d-...
2,2026/06/03 18:05:26,Kemenkes Ungkap Kebiasaan yang Kerap Tak Disad...,416,https://health.detik.com/berita-detikhealth/d-...


In [30]:
print(pilot.loc[0, "url"], "\n")
print(pilot.loc[0, "content"][:1000])

https://health.detik.com/berita-detikhealth/d-8516739/indonesia-jangan-cuma-jadi-pasar-terapi-gen-dunia-kepala-bpom-saatnya-jadi-pemain-utama 

Perkembangan terapi gen, terapi sel, hingga kecerdasan buatan (AI) tengah mengubah wajah dunia kesehatan global. Penyakit yang dulu sulit ditangani, seperti kanker, penyakit genetik langka, hingga kerusakan organ, kini mulai bisa diatasi melalui pendekatan yang lebih presisi dan berbasis teknologi biologis modern.
Kepala Badan Pengawas Obat dan Makanan (BPOM RI) Taruna Ikrar, menilai Indonesia tidak boleh hanya menjadi pasar bagi teknologi kesehatan yang dikembangkan negara lain. Menurutnya, Indonesia harus mulai mengambil peran sebagai pemain utama dalam inovasi biofarmasi modern.
"Transformasi besar sedang terjadi dalam dunia kesehatan global. Indonesia tidak boleh hanya menjadi pasar teknologi kesehatan dunia, tetapi harus mulai mengambil posisi sebagai pemain utama dalam inovasi biofarmasi modern," ujar Taruna dalam keterangannya.
ADVERTISE

### I. Volume Probe

`PER_DAY` must be set by the smallest channel, otherwise that label will fall short while the
others fill their quota. This probe counts available articles per label across 6 sample dates
(3 weekdays, 3 weekend days, spread over the intended period), since weekends publish less.
The results determine `PER_DAY`, `START` and `END` in Section G.

In [31]:
from statistics import median

probe_dates = [date(2026, 6, 3),   # Wed
               date(2026, 6, 6),   # Sat
               date(2026, 6, 7),   # Sun
               date(2026, 7, 15),  # Wed
               date(2026, 8, 1),   # Sat
               date(2026, 8, 19)]  # Wed

probe = {}
for label, bases in CHANNELS.items():
    probe[label] = [len({u for b in bases for u in collect_urls(b, d)}) for d in probe_dates]
    print(f"{label:14s} {probe[label]}  median={median(probe[label]):.0f}  min={min(probe[label])}")

business       [75, 34, 29, 63, 31, 69]  median=48  min=29
technology     [24, 17, 21, 27, 17, 36]  median=22  min=17
sports         [61, 53, 57, 31, 44, 66]  median=55  min=31
health         [18, 17, 17, 18, 17, 14]  median=17  min=14
entertainment  [23, 21, 20, 24, 24, 30]  median=24  min=20


### J. Stage A — Build the URL List

In this section, we'll start our plan on collecting articles from START to END (as we discussed on section G) from that label's
channel(s), remove duplicates by article ID, and randomly sample PER_DAY of them. Results are gonna be appended to 'urls.csv' after each label day fetched, so that we can resume if the process is interrupted.

In [40]:
# every date from START to END, inclusive
days = [START + timedelta(days=n) for n in range((END - START).days + 1)]

# the slice this run will cover - LIMIT is set in Section G
days_run = days if LIMIT is None else days[:LIMIT]

# (date, label) pairs already written, so a re-run can skip them
if URLS_CSV.exists() and URLS_CSV.stat().st_size > 0:
    prev = pd.read_csv(URLS_CSV, dtype=str)
    done = set(zip(prev["date"], prev["label"]))
else:
    done = set()

failures = []

# what is actually left to fetch, counted rather than assumed
todo = [(lbl, d) for lbl in LABELS for d in days_run
        if (d.isoformat(), lbl) not in done]

print(f"{len(days)} days in range ({days[0]} .. {days[-1]}), running {len(days_run)}")
print(f"label-days: {len(days_run) * len(LABELS)} total | done {len(done)} | to do {len(todo)}"
      f" | est. {len(todo) * 8 / 60:.0f} min at ~8 s each")


92 days in range (2026-06-01 .. 2026-08-31), running 92
label-days: 460 total | done 10 | to do 450 | est. 60 min at ~8 s each


In [33]:
days

[datetime.date(2026, 6, 1),
 datetime.date(2026, 6, 2),
 datetime.date(2026, 6, 3),
 datetime.date(2026, 6, 4),
 datetime.date(2026, 6, 5),
 datetime.date(2026, 6, 6),
 datetime.date(2026, 6, 7),
 datetime.date(2026, 6, 8),
 datetime.date(2026, 6, 9),
 datetime.date(2026, 6, 10),
 datetime.date(2026, 6, 11),
 datetime.date(2026, 6, 12),
 datetime.date(2026, 6, 13),
 datetime.date(2026, 6, 14),
 datetime.date(2026, 6, 15),
 datetime.date(2026, 6, 16),
 datetime.date(2026, 6, 17),
 datetime.date(2026, 6, 18),
 datetime.date(2026, 6, 19),
 datetime.date(2026, 6, 20),
 datetime.date(2026, 6, 21),
 datetime.date(2026, 6, 22),
 datetime.date(2026, 6, 23),
 datetime.date(2026, 6, 24),
 datetime.date(2026, 6, 25),
 datetime.date(2026, 6, 26),
 datetime.date(2026, 6, 27),
 datetime.date(2026, 6, 28),
 datetime.date(2026, 6, 29),
 datetime.date(2026, 6, 30),
 datetime.date(2026, 7, 1),
 datetime.date(2026, 7, 2),
 datetime.date(2026, 7, 3),
 datetime.date(2026, 7, 4),
 datetime.date(2026, 7, 5),

In [41]:
for label in LABELS:
    for day in tqdm(days_run, desc=label):          # days_run honours LIMIT from Section G
        if (day.isoformat(), label) in done:        # string vs string - `done` was rebuilt from the CSV
            continue

        try:
            # 1. every URL this label's channel(s) listed that day
            pool = []
            for base in CHANNELS[label]:            # sports has two bases, so one base at a time
                pool += collect_urls(base, day)

            # 2. collapse duplicates by article id (same story filed under two sections)
            unique = {}
            for u in pool:
                m = re.search(r"/d-(\d+)/", u)
                if m:                               # a non-matching URL is skipped, not fatal
                    unique[m.group(1)] = u

            # 3. freeze the order, then sample reproducibly
            ordered = sorted(unique.values())
            n       = min(PER_DAY, len(ordered))    # thin days hold fewer than PER_DAY
            rng     = random.Random(f"{SEED}-{day}-{label}")
            picked  = rng.sample(ordered, n)

            # 4. append this label-day straight away
            rows   = [{"label": label, "date": day.isoformat(), "url": u} for u in picked]
            header = not URLS_CSV.exists()          # evaluated BEFORE the write -> header appears once
            pd.DataFrame(rows).to_csv(URLS_CSV, mode="a", header=header, index=False)

            # 5. only now is the label-day really done
            done.add((day.isoformat(), label))
            tqdm.write(f"{label} {day}: {len(pool)} found, {len(unique)} unique, {n} kept")

        except Exception as e:                      # not a bare `except:` - keeps Ctrl-C working
            failures.append((label, day, e))
            tqdm.write(f"FAILED {label} {day}: {type(e).__name__}: {e}")

print("label-days done:", len(done), "| failures:", len(failures))


business:   3%|▎         | 3/92 [00:15<07:39,  5.16s/it]

business 2026-06-03: 75 found, 75 unique, 14 kept


business:   4%|▍         | 4/92 [00:29<11:54,  8.11s/it]

business 2026-06-04: 84 found, 84 unique, 14 kept


business:   5%|▌         | 5/92 [00:39<12:31,  8.63s/it]

...

entertainment 2026-08-29: 23 found, 23 unique, 14 kept


entertainment:  99%|█████████▉| 91/92 [08:32<00:06,  6.90s/it]

entertainment 2026-08-30: 22 found, 22 unique, 14 kept


entertainment: 100%|██████████| 92/92 [08:39<00:00,  5.64s/it]

entertainment 2026-08-31: 24 found, 24 unique, 14 kept
label-days done: 460 | failures: 0


Result have been checked, resulting 6000+ urls.

### K. Stage B — Download the Articles

Stage A produced `urls.csv`, a list of 6,417 article URLs. Stage B walks that list, calls
`parse_article` on each URL, and appends the result to `articles.csv` — one row per article,
written immediately so an interrupted run keeps everything it already fetched.

At `delay=1.5` per request this is roughly 3.5 hours for the full list, so run it overnight.
`B_LIMIT` in Section G keeps the first pass small: 20 articles to prove the loop, then `None`
for the real run.

Resume works the same way as Stage A, but the key is the **URL** rather than the `(date, label)`
pair, because Stage B's unit of work is one article.

Content is stored **raw** — `SCROLL TO CONTINUE WITH CONTENT`, `ADVERTISEMENT` and the
`detikcom` self-mentions are left in and removed in `02_preprocessing`, so that cleaning
decisions are documented in the notebook that justifies them rather than buried in the scraper.


In [43]:
articles = pd.read_csv(URLS_CSV, dtype=str)
B_LIMIT = None             # articles to download this run: 20 = smoke test, None = all of urls.csv

# urls already downloaded, so a re-run can skip them
if ARTICLES_CSV.exists() and ARTICLES_CSV.stat().st_size > 0:
    prev = pd.read_csv(ARTICLES_CSV, dtype=str)
    got = set(prev["url"])
else:
    got = set()

# what is left to fetch, counted rather than assumed
work = articles[~articles["url"].isin(got)].reset_index(drop=True)
work = work if B_LIMIT is None else work.head(B_LIMIT)

failed_b = []


def fetch_and_append(url, label, index_date):
    """Download one article and append it to articles.csv.

    Returns None on success, or the error text so the caller can log it.
    Only parse_article is guarded: a failure to WRITE should stop the run,
    not be logged 6,000 times.
    """
    try:
        art = parse_article(url)
    except Exception as e:
        return f"{type(e).__name__}: {e}"

    art["label"] = label            # the channel label from urls.csv
    art["index_date"] = index_date  # the index date, kept beside the article's own publishdate

    header = not ARTICLES_CSV.exists()   # evaluated BEFORE the write -> header appears once
    pd.DataFrame([art]).to_csv(ARTICLES_CSV, mode="a", header=header, index=False)
    return None


print(f"urls.csv: {len(articles)} | already downloaded: {len(got)}"
      f" | to fetch now: {len(work)} of {len(articles) - len(got)} remaining")
print(f"est. {len(work) * 2 / 60:.0f} min at ~2 s each")


urls.csv: 6417 | already downloaded: 0 | to fetch now: 6417 of 6417 remaining
est. 214 min at ~2 s each


In [44]:
for r in tqdm(work.itertuples(index=False), total=len(work), desc="articles"):
    err = fetch_and_append(r.url, r.label, r.date)

    if err is None:
        got.add(r.url)                                   # only after the append succeeded
    else:
        failed_b.append({"url": r.url, "label": r.label, "date": r.date, "error": err})
        tqdm.write(f"FAILED {r.url}: {err}")

print("appended:", len(work) - len(failed_b), "| failed:", len(failed_b))


articles:  91%|█████████▏| 5858/6417 [3:38:12<46:16,  4.97s/it]  

FAILED https://hot.detik.com/celeb/d-8586281/sheila-dara-cerita-memori-kebaya-lamaran-dengan-vidi-aldiano: HTTPError: 404 Client Error: Not Found for url: https://hot.detik.com/celeb/d-8586281/sheila-dara-cerita-memori-kebaya-lamaran-dengan-vidi-aldiano


articles: 100%|██████████| 6417/6417 [3:56:56<00:00,  2.22s/it]

appended: 6416 | failed: 1


A failure here is usually a timeout or a 5xx, not a broken URL, so the failed list is worth
one retry before it is written to `failed.csv` for inspection. Run this cell only if
`failed_b` is non-empty.


In [45]:
still_failed = []

for f in tqdm(failed_b, desc="retry"):
    err = fetch_and_append(f["url"], f["label"], f["date"])
    if err is None:
        got.add(f["url"])
    else:
        still_failed.append({**f, "error": err})

if still_failed:
    pd.DataFrame(still_failed).to_csv(FAILED_CSV, index=False)

print("recovered:", len(failed_b) - len(still_failed), "| still failed:", len(still_failed))


retry: 100%|██████████| 1/1 [00:02<00:00,  2.05s/it]

recovered: 1 | still failed: 0


Quick shape check on what landed. Empty and very short articles are **counted, not dropped** —
the minimum-length filter belongs in Task 2, where the threshold can be justified.


In [46]:
arts = pd.read_csv(ARTICLES_CSV, dtype=str)

print("rows:", len(arts))
print("columns:", arts.columns.tolist())
print(arts.groupby("label").size())

body = arts["content"].fillna("")
print("empty content:", (body.str.len() == 0).sum())
print("under 200 chars:", (body.str.len() < 200).sum())
print("median length:", int(body.str.len().median()))
print("duplicate urls:", arts["url"].duplicated().sum())
print("duplicate titles:", arts["title"].duplicated().sum())
print("mentions 'detik' in body:", body.str.contains("detik", case=False).mean().round(3))


rows: 6417
columns: ['url', 'title', 'date', 'author', 'tags', 'content', 'label', 'index_date']
label
business         1284
entertainment    1284
health           1283
sports           1282
technology       1284
dtype: int64
empty content: 3
under 200 chars: 525
median length: 2242
duplicate urls: 0
duplicate titles: 8
mentions 'detik' in body: 0.226
